### Multi-prompt CoT

In [ ]:
import openai
import os
openai.api_key = os.getenv("OPENAI_API_KEY")

def run_prompt(prompt):
    response = openai.ChatCompletion.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
    )
    return response['choices'][0]['message']['content']

user_question = "Identify key factors a company must consider when expanding into a new international market."
countries_to_analyze = ["Brazil", "Germany"]

# --- Step 1: Problem Breakdown Prompt ---
problem_breakdown_prompt = f"""
You are an AI assistant specializing in helping companies develop marketing and business expansion strategies.
{user_question}. List the major categories, such as economic viability, legal regulations,
market demand, competition, and infrastructure.
"""

# Run Step 1
breakdown_response = run_prompt(problem_breakdown_prompt)
print("\nProblem Breakdown Response:\n", breakdown_response)

# Parse factors (simple split)
factors = [line.strip() for line in breakdown_response.split("\n") if line.strip()]

# --- Step 2: Factor Analysis Prompts ---
factor_analysis_results = []

for factor in factors:
    factor_prompt = f"""
    You are analyzing a specific factor related to {user_question}.
    Factor: {factor}

    Compare {countries_to_analyze} countries based on this factor.
    Identify key elements, assess potential risks and opportunities, and provide a concise comparison.
    """
    factor_result = run_prompt(factor_prompt)
    factor_analysis_results.append(factor_result)
    print(factor_result)

# --- Step 3: Final Synthesis Prompt ---

# Prepare input for final synthesis
synthesis_input = "\n".join(factor_analysis_results)

final_synthesis_prompt = f"""
You are an AI assistant specializing in helping companies with marketing and business expansion strategies.
You are provided with the following analyses related to {user_question}:

{synthesis_input}

Based on these analyses, determine which country ({countries_to_analyze}) is better suited for expansion.
Justify your recommendation by carefully weighing economic, legal, and market demand factors.
"""

final_response = run_prompt(final_synthesis_prompt)
print("\nFinal Recommendation:\n", final_response)

### Prompt chaining with LangChain

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI
from langchain.chains import LLMChain, SimpleSequentialChain

# Initialize the language model
llm = OpenAI(temperature=0.5)

# Step 1: Create a prompt template to generate a business idea
idea_prompt = PromptTemplate.from_template(
    "Suggest an innovative business idea in the {industry} industry."
)
idea_chain = LLMChain(llm=llm, prompt=idea_prompt)

# Step 2: Create a prompt template to write a marketing pitch for the business idea
pitch_prompt = PromptTemplate.from_template(
    "Write a short marketing pitch for this business idea: {business_idea}"
)
pitch_chain = LLMChain(llm=llm, prompt=pitch_prompt)

# Step 3: Combine the chains into a sequential flow
chain = SimpleSequentialChain(chains=[idea_chain, pitch_chain])

# Run the chained prompts
result = chain.run("technology")

print(result)